# Discrete Time Models & Cellular Automata

Welcome to the second interactive notebook of the `digital-twins` repository.

Unlike continuous models (which use differential equations and numerical integrators), **Discrete Time Models** update their state at fixed, discrete intervals (time steps). The state transition function $\delta_t$ is defined as a difference equation:

$$ x(t+1) = \delta(x(t), u(t)) $$

### 1D Cellular Automata
A classic example of discrete time simulation is the 1D Cellular Automaton. 
- **Space** is discretized into a grid of cells.
- **Time** advances in fixed steps (generations).
- **State** is discrete (e.g., a cell is either `0` or `1`).
- **Behavior** is dictated by a rule comparing a cell and its immediate neighbors.

**Why this matters for DT:**
Because the state is discrete and transitions via logic rules rather than smooth equations, the resulting behavior is highly non-linear and non-Gaussian. This is why standard Data Assimilation tools like the **Kalman Filter** completely fail on these systems, necessitating the use of **Particle Filters** .

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from ipywidgets import interact, IntSlider

# Ensure Python can find the 'src' directory in the root of the repository

# Import our engine and visualization tools
from digital_twins.models.discrete_time import CellularAutomaton1D, DiscreteTimeSimulator
from digital_twins.visualization.spatial_plots import plot_discrete_grid

### Interactive Learning: Stephen Wolfram's 256 Rules

In the 1980s, Stephen Wolfram discovered that by looking at a cell and its two neighbors (3 cells total), there are $2^3 = 8$ possible neighborhood patterns. If we assign a resulting state (`0` or `1`) to each of those 8 patterns, we get $2^8 = 256$ possible rule sets for the universe.

**Instructions:**
Use the slider below to change the `Rule Number` (0-255). Watch how the simulation output completely transforms. 

*Try these famous rules:*
- **Rule 30:** It creates chaotic, pseudorandom patterns. It is so mathematically complex it is used as a random number generator.
- **Rule 90:** Generates the famous Sierpinski triangle fractal.
- **Rule 110:** Proven to be Turing complete (capable of universal computation)!
- **Rule 184:** Often used as a simple model for 1D traffic flow.

In [ ]:
def interactive_cellular_automaton(rule_number=30, time_steps=40):
    # 1. To keep the pyramid from hitting the walls, we size the grid based on time steps
    grid_size = time_steps * 2 + 1
    
    # 2. Initialize the Model and Simulator
    ca_model = CellularAutomaton1D(rule_number=rule_number)
    simulator = DiscreteTimeSimulator(model=ca_model)
    
    # 3. Create initial state: All zeros with a single 1 in the dead center
    x0 = np.zeros(grid_size, dtype=int)
    x0[grid_size // 2] = 1
    
    # 4. Run the Discrete Time Simulation
    t_hist, x_hist, _ = simulator.simulate(
        t_start=0, 
        t_end=time_steps, 
        x0=x0
    )
    
    # 5. Plotting the results
    fig, ax = plt.subplots(figsize=(10, 6))
    bw_cmap = ListedColormap(['white', 'black'])
    
    plot_discrete_grid(
        grid=x_hist, 
        cmap=bw_cmap, 
        ax=ax, 
        title=f"1D Cellular Automata | Rule {rule_number} | {time_steps} Generations"
    )
    
    # Restore some axes info for educational clarity
    ax.set_ylabel("Time Step (t)")
    ax.set_xlabel("1D Spatial Cell Index")
    
    # Add a border
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)
        
    plt.tight_layout()
    plt.show()

# Create the interactive widget
interact(interactive_cellular_automaton, 
         rule_number=IntSlider(value=30, min=0, max=255, step=1, description='Rule Number:', continuous_update=False),
         time_steps=IntSlider(value=50, min=10, max=200, step=10, description='Time Steps:', continuous_update=False))